## Groq Explanations (RAG + Kill Switch)

Generates plain-English compliance reports for each flagged prescriber by combining the RAG regulatory retrieval pipeline with Llama 3.3 70B via the Groq API. Implements a hard Kill Switch to prevent LLM calls when no sufficiently relevant regulatory reference is found.

### What this notebook does
- Loads `pharmaguard_results.csv` and selects the top 50 flagged cases by anomaly score
- For each flagged case, builds a semantic query based on the detected fraud type
- Retrieves the most relevant regulatory chunk from the FAISS index (EMA/FDA documents)
- Checks the FAISS L2 distance against a hard threshold (`KILL_SWITCH_THRESHOLD = 1.0`):
  - If **distance > 1.0** → Kill Switch activated, Groq API is **not called**, returns manual review message
  - If **distance ≤ 1.0** → calls Llama 3.3 70B via Groq API to generate a structured compliance report
- Includes automatic retry logic (up to 5 attempts) with progressive backoff on rate limit errors
- Outputs `pharmaguard_explained.csv` with explanations and Kill Switch metadata

### Kill Switch logic
```python
if distance > KILL_SWITCH_THRESHOLD:
    explanation = "⚠️ ANOMALY MATHEMATICALLY DETECTED — NO REGULATORY MATCH FOUND — MANUAL REVIEW REQUIRED."
else:
    explanation = generate_explanation(row, regulation, reg_source)
```

### Why 50 cases
The Groq free tier has a daily limit of 100,000 tokens. At ~2,000 tokens per explanation, 50 cases consume approximately 100,000 tokens. This covers all cases visible in the Streamlit dashboard (top 50 by anomaly score) without exceeding the daily quota.

### Requirements
- A valid Groq API key in a `.env` file: `GROQ_API_KEY=your_key_here`
- `pharmaguard_results.csv` (output of `isolation_forest.ipynb`)
- `pharmaguard_index.faiss` + `pharmaguard_chunks.pkl` (output of `rag_pipeline.ipynb`)

### Output
| File | Description |
|------|-------------|
| `pharmaguard_explained.csv` | Top 50 flagged cases with regulatory references, FAISS distances, Kill Switch status, and Groq explanations |

### Key columns in output
| Column | Description |
|--------|-------------|
| `faiss_distance` | L2 distance of retrieved regulatory chunk — lower = more relevant |
| `reg_confidence` | `OK` if Groq was called, `BLOCKED` if Kill Switch activated |
| `groq_called` | Boolean — whether the Groq API was invoked |
| `explanation` | Plain-English compliance report or Kill Switch message |

In [ ]:
import os
import time
import pandas as pd
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

# Config
TOP_K = 1
KILL_SWITCH_THRESHOLD = 1.0

# Load data
df_results = pd.read_csv("pharmaguard_results.csv")
flagged = df_results[df_results['predicted_fraud'] == 1].head(50).copy()

index = faiss.read_index("pharmaguard_index.faiss")
with open("pharmaguard_chunks.pkl", "rb") as f:
    data = pickle.load(f)
texts   = data['texts']
sources = data['sources']

embedder = SentenceTransformer('all-MiniLM-L6-v2')

# RAG retrieval with distance
def retrieve_regulation(query, top_k=TOP_K):
    """
    Returns (chunk_text, source, distance).
    L2 distance: lower = more similar.
    """
    q_emb = embedder.encode([query]).astype('float32')
    distances, indices = index.search(q_emb, top_k)
    idx      = indices[0][0]
    distance = float(distances[0][0])
    return texts[idx][:400], sources[idx], distance


# Query builder
def build_query(row):
    fraud_type = row['fraud_type']
    drug       = row['generic_name']
    specialty  = row['specialty']
    if fraud_type == 'volume_fraud':
        return f"abnormal high prescription volume reporting requirements {drug}"
    elif fraud_type == 'cost_fraud':
        return f"abnormal reimbursement cost fraud pharmaceutical {drug}"
    elif fraud_type == 'off_label':
        return f"off-label prescription {drug} {specialty} reporting requirements"
    else:
        return f"suspicious prescription pattern {drug}"

# Groq explanation with retry
def generate_explanation(row, regulation, regulation_source):
    prompt = f"""You are a pharmaceutical compliance AI assistant.
Analyze the following suspicious prescription case and generate a clear,
concise explanation for a compliance officer.

CASE DETAILS:
- Drug: {row['generic_name']}
- Prescriber Specialty: {row['specialty']}
- Total Claims: {int(row['total_claims'])}
- Total Cost: ${float(row['total_cost']):,.2f}
- Total Patients: {int(row['total_patients'])}
- Anomaly Score: {float(row['anomaly_score']):.4f}
- Fraud Type Detected: {row['fraud_type']}

RELEVANT REGULATORY REFERENCE ({regulation_source}):
{regulation}

Generate a compliance report with:
1. A one-sentence summary of why this case is suspicious
2. The specific fraud pattern detected
3. The regulatory violation based on the reference above
4. Recommended action for the compliance officer

Keep it concise and professional. Maximum 150 words."""

    for attempt in range(5):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=300,
                temperature=0.3
            )
            return response.choices[0].message.content
        except Exception as e:
            if "429" in str(e):
                wait = 60 * (attempt + 1)
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/5...")
                time.sleep(wait)
            else:
                raise e

    return "⚠️ Rate limit exceeded — explanation not generated. Re-run tomorrow."

# Kill Switch message
KILL_SWITCH_MESSAGE = (
    "⚠️ ANOMALY MATHEMATICALLY DETECTED — NO REGULATORY MATCH FOUND — "
    "MANUAL REVIEW REQUIRED. "
    "The anomaly detection model flagged this prescriber, but no sufficiently "
    "relevant regulatory reference was found in the knowledge base "
    "(FAISS distance > threshold). "
    "A compliance officer must review this case independently without AI guidance."
)

# Main loop 
print(f"Processing {len(flagged)} flagged cases")
print(f"Kill Switch threshold: {KILL_SWITCH_THRESHOLD}")
print("=" * 70)

results = []

for i, (_, row) in enumerate(flagged.iterrows()):
    print(f"[{i+1}/{len(flagged)}] {row['generic_name']} — {row['fraud_type']}", end=" ... ")

    query = build_query(row)
    regulation, reg_source, distance = retrieve_regulation(query)

    # Kill Switch 
    if distance > KILL_SWITCH_THRESHOLD:
        explanation    = KILL_SWITCH_MESSAGE
        reg_confidence = "BLOCKED"
        groq_called    = False
        print(f"BLOCKED (distance={distance:.3f})")
    else:
        explanation    = generate_explanation(row, regulation, reg_source)
        reg_confidence = "OK"
        groq_called    = True
        print(f"OK (distance={distance:.3f})")
    # Append results

    results.append({
        **row.to_dict(),
        'query'            : query,
        'regulation_text'  : regulation,
        'regulation_source': reg_source,
        'faiss_distance'   : round(distance, 4),
        'reg_confidence'   : reg_confidence,
        'groq_called'      : groq_called,
        'explanation'      : explanation,
    })

df_explained = pd.DataFrame(results)

# Stats
total   = len(df_explained)
blocked = len(df_explained[df_explained['reg_confidence'] == 'BLOCKED'])
explained = total - blocked

print(f"\nTotal cases processed : {total}")
print(f"Explained by Groq     : {explained}  ({explained/total*100:.1f}%)")
print(f"Blocked (Kill Switch) : {blocked}  ({blocked/total*100:.1f}%)")
print(f"\nFAISS distance stats:")
print(df_explained['faiss_distance'].describe().round(4).to_string())

# Save
df_explained.to_csv("pharmaguard_explained.csv", index=False)
print(f"\nSaved: pharmaguard_explained.csv")
print(f"Columns: {list(df_explained.columns)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Processing 50 flagged cases
Kill Switch threshold: 1.0
[1/50] Rsvpref3 Antigen/As01e/Pf — legitimate ... BLOCKED (distance=1.163)
[2/50] Alprazolam — cost_fraud ... OK (distance=0.945)
[3/50] Ondansetron Hcl — cost_fraud ... BLOCKED (distance=1.022)
[4/50] Diclofenac Sodium — cost_fraud ... OK (distance=0.972)
[5/50] Leflunomide — volume_fraud ... BLOCKED (distance=1.080)
[6/50] Semaglutide — legitimate ... BLOCKED (distance=1.243)
[7/50] Semaglutide — off_label ... OK (distance=0.894)
[8/50] Nitrofurantoin Monohyd/M-Cryst — legitimate ... BLOCKED (distance=1.327)
[9/50] Apixaban — cost_fraud ... OK (distance=0.900)
[10/50] Alprazolam — cost_fraud ... OK (distance=0.945)
[11/50] Spironolactone — cost_fraud ... BLOCKED (distance=1.066)
[12/50] Atorvastatin Calcium — legitimate ... BLOCKED (distance=1.314)
[13/50] Carvedilol — cost_fraud ... OK (distance=0.907)
[14/50] Cephalexin — cost_fraud ... BLOCKED (distance=1.041)
[15/50] Dulaglutide — legitimate ... BLOCKED (distance=1.263)
[16/5